This Notebook uses constructed data offline. Fixed filing facts and reference figures are historical teaching materials; new calculations do not reproduce a market strategy.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch07_statistics_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


# Chapter 7: estimation, inference, and bootstrap

This walkthrough uses **constructed teaching samples only**. It makes no market-performance claim. The sample unit is a return per constructed observation period. The fixed master seed is `20261007`; inference and resampling outputs are reproducible in the local course environment.

## Goal

Compare a Gaussian maximum-likelihood variance with the `ddof=1` estimate, form a Student-t interval and p-value, simulate interval coverage, and compare IID with fixed-length moving-block bootstrap means for a constructed AR(1) series. The bootstrap receives only a chronological training slice. A fixed block length is illustrative and does not guarantee valid inference.


## Observed run (seed 20261007)

The 40-observation constructed normal sample has mean −0.001500, MLE variance 0.00045812, and `ddof=1` variance 0.00046987. Its 95% Student-t interval is [−0.008432, 0.005432] return per constructed observation period; the two-sided p-value against zero is 0.664051. In the 3,000-repetition coverage simulation, observed coverage was 95.33%, 94.93%, and 95.50% for n = 10, 30, and 100. For the AR(1) training slice (n = 180, φ = 0.70), the IID percentile interval was [−0.002327, 0.001643] and the fixed-12 moving-block interval was [−0.003428, 0.002961]. These are fixed-seed constructed results, not market evidence.


## 1. Setup and shared implementation

Core estimators, tests, simulation, and resampling are imported from `src/quant_course/statistics_tools.py`; the Notebook does not duplicate that logic.

In [ ]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src" / "quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from quant_course.statistics_tools import (
    beta_binomial_posterior,
    bootstrap_mean_confidence_interval,
    bootstrap_mean_draws,
    mean_t_confidence_interval,
    moving_block_bootstrap_mean_draws,
    normal_mle_estimates,
    one_sample_t_test,
    simulate_stationary_ar1,
    simulate_t_interval_coverage,
)

SEED = 20261007
PERIOD_UNIT = "return per constructed observation period"
seed_sequence = np.random.SeedSequence(SEED)
normal_seed, coverage_seed, ar_seed, iid_seed, block_seed = seed_sequence.spawn(5)
normal_rng = np.random.default_rng(normal_seed)
coverage_rng = np.random.default_rng(coverage_seed)
ar_rng = np.random.default_rng(ar_seed)
iid_rng = np.random.default_rng(iid_seed)
block_rng = np.random.default_rng(block_seed)


## 2. MLE, an unbiased variance estimate, and a t interval

The MLE variance for a normal sample divides squared deviations by \(n\); the usual unbiased sample variance divides by \(n-1\). The t interval uses the sample standard deviation and \(n-1\) degrees of freedom.

In [ ]:
sample_n = 40
assumed_teaching_mean = 0.002
assumed_teaching_std = 0.02
normal_sample = normal_rng.normal(assumed_teaching_mean, assumed_teaching_std, size=sample_n)
estimates = normal_mle_estimates(normal_sample)
interval = mean_t_confidence_interval(normal_sample, confidence_level=0.95)
test = one_sample_t_test(normal_sample, null_mean=0.0)

print(f"CONSTRUCTED normal sample: n={sample_n}, seed={SEED}, unit={PERIOD_UNIT}")
print(f"sample mean={estimates.mean:.6f}")
print(f"MLE variance (ddof=0)={estimates.variance_mle:.8f}; unbiased variance (ddof=1)={estimates.variance_unbiased:.8f}")
print(f"95% Student-t mean CI=[{interval.lower:.6f}, {interval.upper:.6f}] {PERIOD_UNIT}; df={sample_n - 1}")
print(f"two-sided t={test.t_statistic:.6f}, p={test.p_value:.6f}")
print("The p-value is a tail probability under H0 and the test assumptions; it is not the probability that H0 is true.")

hand_check = normal_mle_estimates([1.0, 2.0, 3.0])
assert np.isclose(hand_check.variance_mle, 2.0 / 3.0)
assert hand_check.variance_unbiased == 1.0
assert interval.lower < interval.estimate < interval.upper


## 3. A Beta-Binomial update

With prior Beta(2, 2), six successes and four failures give Beta(8, 6), whose posterior mean is 4/7. This model estimates a Bernoulli success probability only. It does not specify payoff size, loss size, or expected value.

In [ ]:
posterior = beta_binomial_posterior(6, 4, prior_alpha=2.0, prior_beta=2.0)
print(f"prior Beta({posterior.prior_alpha:g}, {posterior.prior_beta:g}); successes={posterior.successes}; failures={posterior.failures}")
print(f"posterior Beta({posterior.posterior_alpha:g}, {posterior.posterior_beta:g}); posterior mean={posterior.posterior_mean:.6f}")
assert (posterior.posterior_alpha, posterior.posterior_beta) == (8.0, 6.0)
assert np.isclose(posterior.posterior_mean, 4.0 / 7.0)


## 4. Coverage under repeated normal sampling

Each repetition draws an independent normal sample with the stated population mean and standard deviation, then records whether the 95% Student-t interval covers the known mean. These are Monte Carlo teaching results; the finite repetition count causes simulation error.

In [ ]:
coverage_repetitions = 3000
coverage_sizes = [10, 30, 100]
true_mean = 0.001
true_std = 0.02
coverage = simulate_t_interval_coverage(
    coverage_sizes,
    repetitions=coverage_repetitions,
    true_mean=true_mean,
    true_std=true_std,
    confidence_level=0.95,
    rng=coverage_rng,
)
print(f"CONSTRUCTED normal coverage simulation: true mean={true_mean}, true sd={true_std} {PERIOD_UNIT}")
print(f"seed={SEED}; repetitions per n={coverage_repetitions}; nominal coverage=95%; observed rates={coverage}")


## 5. IID and moving-block bootstrap on a dependent series

Generate a stationary Gaussian AR(1) teaching series, with \(\phi=0.70\). The first 180 observations are the training slice supplied to both bootstrap routines; the remaining observations are held out and are never used to form the bootstrap distributions. The moving-block implementation samples circular blocks and truncates each resample to the training length. Its fixed block length is not a general validity guarantee. Applicability requires a suitable stationary, weakly dependent process and a defensible block-size choice.

In [ ]:
full_n = 240
train_n = 180
phi = 0.70
innovation_std = 0.01
block_length = 12
bootstrap_repetitions = 4000
ar_series = simulate_stationary_ar1(full_n, phi=phi, innovation_std=innovation_std, rng=ar_rng)
training = ar_series[:train_n]
held_out = ar_series[train_n:]
iid_draws = bootstrap_mean_draws(training, n_resamples=bootstrap_repetitions, rng=iid_rng)
block_draws = moving_block_bootstrap_mean_draws(
    training,
    block_length=block_length,
    n_resamples=bootstrap_repetitions,
    rng=block_rng,
)
iid_ci = bootstrap_mean_confidence_interval(iid_draws)
block_ci = bootstrap_mean_confidence_interval(block_draws)
effective_n_approx = train_n * (1 - phi) / (1 + phi)
print(f"CONSTRUCTED stationary AR(1): n={full_n}, phi={phi:.2f}, innovation sd={innovation_std:.3f} {PERIOD_UNIT}")
print(f"training n={len(training)}, held-out n={len(held_out)}; bootstrap repetitions={bootstrap_repetitions}; fixed block length={block_length}")
print(f"approximate effective sample size n(1-phi)/(1+phi)={effective_n_approx:.2f} (heuristic under AR(1) assumptions)")
print(f"IID percentile 95% CI=[{iid_ci.lower:.6f}, {iid_ci.upper:.6f}] {PERIOD_UNIT}")
print(f"moving-block percentile 95% CI=[{block_ci.lower:.6f}, {block_ci.upper:.6f}] {PERIOD_UNIT}")
assert iid_ci.lower < iid_ci.upper
assert block_ci.lower < block_ci.upper


## 6. Visual checks

The left panel reports empirical coverage by sample size, with the nominal 95% level as a reference. The right panel compares two bootstrap distributions of the training-sample mean in the constructed AR(1) example. Their difference illustrates how resampling assumptions affect uncertainty; it does not establish that one method is valid for every financial series.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sizes = np.array(list(coverage))
rates = np.array([coverage[size] for size in sizes])
axes[0].plot(sizes, rates * 100.0, marker="o", color="#235789")
axes[0].axhline(95.0, color="#C1292E", linestyle="--", label="Nominal 95%")
axes[0].set_xlabel("Sample size n (constructed observations)")
axes[0].set_ylabel("Intervals covering true mean (%)")
axes[0].set_title(f"Student-t interval coverage; {coverage_repetitions:,} simulations")
axes[0].set_ylim(85, 100)
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].hist(iid_draws, bins=45, density=True, alpha=0.52, label="IID bootstrap")
axes[1].hist(block_draws, bins=45, density=True, alpha=0.52, label=f"Moving blocks (L={block_length})")
axes[1].axvline(float(iid_draws.mean()), color="#235789", linestyle=":", linewidth=1.5)
axes[1].set_xlabel("Bootstrap mean (return per constructed period)")
axes[1].set_ylabel("Density (inverse return unit)")
axes[1].set_title(f"AR(1) training n={train_n}, phi={phi:.2f}; {bootstrap_repetitions:,} draws")
axes[1].grid(alpha=0.2)
axes[1].legend(frameon=False)
fig.suptitle(f"Chapter 7 — constructed simulation only; master seed {SEED}; no empirical performance inference")
fig.tight_layout()
figure_path = RUN_DIR / "figures" / "a2" / "ch07_statistics.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.close(fig)
from IPython.display import Image, display
display(Image(filename=str(figure_path)))
print(f"Saved figure: {figure_path.relative_to(ROOT)}")


## Checks and limitations

- The hand check gives MLE variance 2/3 and unbiased variance 1 for `[1, 2, 3]`.
- The Beta-Binomial update is Beta(8, 6), mean 4/7.
- NaN, infinite, complex, non-1D, and too-short samples are rejected instead of silently imputed. A constant sample is valid for descriptive variance but rejected for t inference because its estimated standard error is zero.
- A p-value is conditional on the null and test assumptions; it is not the probability the null is true. Type I/II error and power depend on a specified repeated-sampling design; multiple comparisons change error behavior and require a declared family of tests.
- IID resampling breaks serial order. Fixed-length moving blocks preserve local order but need stationarity/weak-dependence assumptions, a considered block-length choice, and sensitivity checks in real work. The AR(1) effective-sample-size formula shown above is a model-based approximation.
- The Bayesian example describes only a success probability. Payoff magnitudes and losses are needed to assess expected value.
